# Допълнително упражнение 1А: практика с Python


**Машинно самообучение · зимен семестър 2026/2027**

**Repo:** https://github.com/mkorudzhiev/ML_26_27<br>
**Discord:** https://discord.gg/WKztMqX46

Мартин Коруджиев<br/>
mkorudzhiev@uni-sofia.bg<br/>
0885 71 31 60

## Съдържание

1. [Част 1. От CSV до Python обекти](#chast-1)
2. [Част 2. Условия, цикли и филтриране](#chast-2)
3. [Част 3. Функции и обобщения](#chast-3)
4. [Част 4. Кратък отчет](#chast-4)

<a id="chast-1"></a>
## Част 1. От CSV до Python обекти

### 1. Четем файла

CSV е текстова таблица: запетаите разделят колоните. Първият ред е с имената на колоните,
а всеки следващ ред описва един имот. `csv.DictReader` използва заглавията като ключове
в речник. Първоначално всички стойности са низове. `Path` описва пътя до файла, а `with`
го затваря след прочитането.

In [ ]:
import csv
from pathlib import Path

data_path = Path("data/housing_prices.csv")
with data_path.open(encoding="utf-8", newline="") as file:
    rows = list(csv.DictReader(file))

print("Брой редове:", len(rows))
print("Първи ред:", rows[0])
print("Тип на площта:", type(rows[0]["area_m2"]).__name__)

### 2. Преобразуваме стойностите

За смятане ни трябват числа. Създаваме нов списък от речници:
`float` превръща двата низа в числа. Така не променяме прочетените
текстови редове и можем да сравним двата вида данни.

In [ ]:
homes = [
    {"area_m2": float(row["area_m2"]),
     "price_eur": float(row["price_eur"])}
    for row in rows
]

print("Първи имот:", homes[0])
print("Първа цена + 1000:", homes[0]["price_eur"] + 1000)
print("Оригиналният текст:", rows[0]["price_eur"])

> **Въпрос.** Какво ще отпечата този нов код? Защо промяната на
> `copy_of_list` засяга `values`, а промяната на `another_list` не го засяга?

```python
values = [10, 20, 30]
copy_of_list = values
another_list = values[:]
copy_of_list[0] = 99
another_list[1] = 77
print(values, another_list)
```

<details><summary>Отговор</summary>

Получаваме `[99, 20, 30] [10, 77, 30]`. Присвояването запазва
препратка към същия списък. Срезът `[:]` прави нов списък преди
промяната на `copy_of_list`.

</details>

<a id="chast-2"></a>
## Част 2. Условия, цикли и филтриране

### 3. Пробваме условие върху редовете

Речникът пази площта и цената заедно. При отсяване връщаме цели
речници, за да не разместим площите спрямо цените им.

In [ ]:
large_homes = []
for home in homes:
    if home["area_m2"] >= 80:
        large_homes.append(home)

print("Имотите от поне 80 m² са:", len(large_homes))
print("Техните площи:", [home["area_m2"] for home in large_homes])

#### Нека опитаме 1

Пишем функция `select_homes(homes, min_area, max_price)`, която
връща нов списък с имотите, за които **едновременно**
`area_m2 >= min_area` и `price_eur <= max_price`. Запазваме
първоначалния ред. Използваме `for` и `if` или list comprehension.

Преди проверката предвиждаме колко реда ще получим при
`min_area=80` и `max_price=220000`.

In [ ]:
# TODO: Дописваме функцията.
# def select_homes(homes, min_area, max_price):
#     ...

In [ ]:
if "select_homes" in globals():
    selected = select_homes(homes, min_area=80, max_price=220_000)
    assert [home["area_m2"] for home in selected] == [90.0, 85.0]
    assert select_homes(homes, 200, 500_000) == []
    print("Вярно: площите са", [home["area_m2"] for home in selected])
else:
    print("Попълваме функцията и изпълняваме проверката пак.")

<details><summary>Един възможен отговор</summary>

```python
def select_homes(homes, min_area, max_price):
    return [
        home for home in homes
        if home["area_m2"] >= min_area
        and home["price_eur"] <= max_price
    ]
```

И двете граници са включени. Условието е за **един и същ** имот.

</details>

> **Въпрос.** Без да изпълняваме кода, кои площи остават?
> Защо списъкът `areas` не се променя?

```python
areas = [45, 80, 95, 80]
selected_areas = [area for area in areas if area >= 80]
selected_areas.append(120)
print(areas, selected_areas)
```

<details><summary>Отговор</summary>

Получаваме `[45, 80, 95, 80] [80, 95, 80, 120]`.
Comprehension създава нов списък, а `append` променя само него.

</details>

<a id="chast-3"></a>
## Част 3. Функции и обобщения

### 4. Изчисляваме цена на квадратен метър

Функцията взема един имот и връща число. Ако площта е нула или
отрицателна, делението няма смисъл: съобщаваме това с `ValueError`.
Проверката на входа е част от поведението на функцията.

In [ ]:
def price_per_m2(home):
    area = home["area_m2"]
    if area <= 0:
        raise ValueError("Площта трябва да е положителна.")
    return home["price_eur"] / area

print(f"Първи имот: {price_per_m2(homes[0]):.0f} евро/m²")
try:
    price_per_m2({"area_m2": 0, "price_eur": 100_000})
except ValueError as error:
    print("Очаквана грешка:", error)

### 5. Съставяме речник с обобщения

За цял списък ще върнем един речник. `sum` събира числата,
`len` дава броя им. Ако списъкът е празен, средната стойност
не е определена и функцията трябва да вдигне `ValueError`.

#### Нека опитаме 2

Пишем `summarize_homes(homes)`. Връщаме речник с ключове
`"count"`, `"mean_area"` и `"mean_price"`. Средните са
аритметични средни за подадените имоти. Проверяваме и за празен
списък. Можем да използваме временни списъци или генератори в `sum`.

In [ ]:
# TODO: Дописваме функцията.
# def summarize_homes(homes):
#     ...

In [ ]:
if "summarize_homes" in globals():
    summary = summarize_homes(homes)
    assert set(summary) == {"count", "mean_area", "mean_price"}
    assert summary["count"] == 12
    assert abs(summary["mean_price"] - 181_250) < 1e-9
    assert summarize_homes(homes[:2])["mean_area"] == 46
    try:
        summarize_homes([])
    except ValueError:
        print("Вярно: празният списък няма средна стойност.")
    else:
        raise AssertionError("При празен списък очакваме ValueError.")
    print(summary)
else:
    print("Попълваме функцията и изпълняваме проверката пак.")

<details><summary>Един възможен отговор</summary>

```python
def summarize_homes(homes):
    if not homes:
        raise ValueError("Няма имоти за обобщение.")
    return {
        "count": len(homes),
        "mean_area": sum(home["area_m2"] for home in homes) / len(homes),
        "mean_price": sum(home["price_eur"] for home in homes) / len(homes),
    }
```

Условието `if not homes` проверява дали списъкът е празен.

</details>

<a id="chast-4"></a>
## Част 4. Кратък отчет

### 6. Подреждаме имотите

`sorted` връща нов списък. Аргументът `key` казва по каква
стойност да сравняваме елементите. Подаваме вече написаната
функция `price_per_m2`, така че всеки речник се сравнява по
цена на квадратен метър.

In [ ]:
ranked = sorted(homes, key=price_per_m2)
for home in ranked[:3]:
    print(f'{home["area_m2"]:.0f} m²: {price_per_m2(home):.0f} евро/m²')

print("Първоначалният първи имот още е:", homes[0]["area_m2"], "m²")

#### Нека опитаме 3

Пишем `make_report(homes)`, която връща **низ** с два реда:
`"Имоти: N"` и `"Поне 80 m²: K"`, разделени с `\n`.
`N` е броят на всички имоти, а `K` е броят на имотите с площ
поне 80 m². Не отпечатваме във функцията: тя само връща текста.
Проверяваме я и върху нов, кратък списък.

In [ ]:
# TODO: Дописваме функцията.
# def make_report(homes):
#     ...

In [ ]:
if "make_report" in globals():
    assert make_report(homes) == "Имоти: 12\nПоне 80 m²: 5"
    sample = [
        {"area_m2": 79, "price_eur": 100_000},
        {"area_m2": 80, "price_eur": 120_000},
    ]
    assert make_report(sample) == "Имоти: 2\nПоне 80 m²: 1"
    print(make_report(homes))
else:
    print("Попълваме функцията и изпълняваме проверката пак.")

<details><summary>Един възможен отговор</summary>

```python
def make_report(homes):
    return f"Имоти: {len(homes)}\nПоне 80 m²: {sum(home["area_m2"] >= 80 for home in homes)}"
```

В Python `True` се брои като 1, а `False` като 0. Можем да
напишем същото и с обикновен цикъл и брояч.

</details>